##Demanda real

In [0]:
!pip install pydataxm

In [0]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

from pydataxm.pydatasimem import ReadSIMEM


bronze_table = "observatorio_dev.bronze.demanda_real"
historical_start_date = date(2026, 1, 1)
lookback_days = 45

fecha_fin = datetime.now(
    ZoneInfo("America/Bogota")
).date()

bronze_is_empty = len(
    spark.table(bronze_table).head(1)
) == 0

if bronze_is_empty:
    fecha_inicio = historical_start_date
    execution_mode = "BACKFILL"
else:
    fecha_inicio = fecha_fin - timedelta(days=lookback_days)
    execution_mode = "INCREMENTAL"

fecha_inicio_str = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_str = fecha_fin.strftime("%Y-%m-%d")

print("Modo de ejecución:", execution_mode)
print("Tabla Bronze:", bronze_table)
print(
    f"Rango solicitado a SIMEM: "
    f"{fecha_inicio_str} a {fecha_fin_str}"
)

df_demanda = ReadSIMEM(
    "14FABB",
    fecha_inicio_str,
    fecha_fin_str
).main(filter=False)

if df_demanda is None or df_demanda.empty:
    raise ValueError(
        "SIMEM no devolvió demanda para el rango solicitado"
    )

print(f"Registros descargados: {len(df_demanda):,}")

df_demanda.to_json(
    "/Volumes/observatorio_dev/landing/raw_files/"
    "demanda_real.json",
    orient="records",
    lines=True,
    mode="w"
)

In [0]:
# Guardar como un solo archivo JSON usando pandas, sobrescribiendo si ya existe
df_demanda.to_json("/Volumes/observatorio_dev/landing/raw_files/demanda_real.json", orient='records', lines=True, mode='w')
# Para 10,000 filas, guardar en un solo archivo JSON es eficiente y adecuado; no es necesario dividir en varios archivos.

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    MIN(TO_TIMESTAMP(fecha_hora)) AS fecha_minima,
    MAX(TO_TIMESTAMP(fecha_hora)) AS fecha_maxima,
    COUNT(DISTINCT version) AS versiones_distintas,
    COUNT(DISTINCT codigo_variable) AS variables_distintas,
    COUNT(DISTINCT tipo_mercado) AS mercados_distintos,
    COUNT(DISTINCT codigo_sic_agente) AS agentes_distintos
FROM observatorio_dev.bronze.demanda_real;

In [0]:
%sql
SELECT
    COUNT(*) AS grupos_duplicados,
    COALESCE(SUM(repeticiones), 0) AS filas_en_grupos,
    COALESCE(SUM(repeticiones - 1), 0) AS filas_excedentes,
    COALESCE(MAX(repeticiones), 0) AS maximo_repeticiones
FROM (
    SELECT
        codigo_variable,
        fecha_hora,
        codigo_sic_agente,
        tipo_mercado,
        version,
        valor,
        unidad_medida,
        codigo_duracion,
        COUNT(*) AS repeticiones
    FROM observatorio_dev.bronze.demanda_real
    GROUP BY
        codigo_variable,
        fecha_hora,
        codigo_sic_agente,
        tipo_mercado,
        version,
        valor,
        unidad_medida,
        codigo_duracion
    HAVING COUNT(*) > 1
);